# Evaluation

Evaluate RBM vs baselines using ranking metrics.

## Paper formula (Eq. 34): Hyperbolic activation (distribution view)

We sample multiple movies and look at the hyperbolic activation outputs to see the range of hidden-like features induced by the formula.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ratings_path = project_root / "data" / "rating.csv"
if not ratings_path.exists():
    ratings_path = project_root / "data" / "ratings.csv"

scores_path = project_root / "data" / "genome_scores.csv"
ratings_df = pd.read_csv(ratings_path, usecols=["movieId", "rating"])
movie_rating = ratings_df.groupby("movieId")["rating"].mean().rename("avg_rating")

scores_df = pd.read_csv(scores_path, usecols=["movieId", "tagId", "relevance"])
tag_id = int(scores_df["tagId"].mode().iloc[0])
score_slice = scores_df[scores_df["tagId"] == tag_id].set_index("movieId")

merged = movie_rating.to_frame().join(score_slice[["relevance"]], how="inner")
merged = merged.sample(n=500, random_state=42)

x = (merged["avg_rating"] / 5.0).values + 0.1
raw_y = merged["relevance"].values
scale = np.minimum(0.9 * np.abs(x) / (np.abs(raw_y) + 1e-8), 1.0)
y = raw_y * scale
modulus = np.sqrt(np.maximum(x**2 - y**2, 1e-12))

gx = x / modulus
gy = y / modulus

plt.figure(figsize=(5, 3))
plt.hist(gx, bins=30, alpha=0.7, label="g(z) real")
plt.hist(gy, bins=30, alpha=0.7, label="g(z) unipotent")
plt.title("Hyperbolic activation outputs")
plt.legend()
plt.tight_layout()
plt.show()

In [2]:
import os
import sys
from pathlib import Path

# Resolve project root regardless of where the notebook runs
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.append(str(project_root / "src"))

import torch
from data_prep import DataConfig, load_movielens
from metrics import evaluate_model
from rbm import RBM

config = DataConfig(data_dir=str(project_root / "data"), max_users=2000, max_items=4000)
data = load_movielens(config)
train_matrix = data["train_matrix"]
test_matrix = data["test_matrix"]

rbm = RBM(n_visible=train_matrix.shape[1], n_hidden=128, k=5)

def rbm_scores(user_idx):
    v0 = torch.from_numpy(train_matrix[user_idx].toarray()).float()
    with torch.no_grad():
        return rbm.reconstruct(v0).cpu().numpy().ravel()

print(evaluate_model(rbm_scores, test_matrix, train_matrix, k=20))


{'precision_at_k': 0.0020822880080280984, 'recall_at_k': 0.004250637212475055, 'ndcg_at_k': 0.002799781838800685, 'coverage': 0.009182209469153515}
